In [6]:
!pip install ultralytics

import cv2
import numpy as np
import time
from ultralytics import YOLO

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
# Set to True for WebCam (Option C), False for Video File (Option A or B)
USE_WEBCAM = False

# If USE_WEBCAM is False, specify your input video file path here
INPUT_VIDEO_PATH = "input_squat.mp4"

# The name of the output video file for submission (Deliverable 1)
OUTPUT_VIDEO_PATH = "output_squat_result.mp4"
# ==========================================

def calculate_angle(p1, p2, p3):
    """
    Calculate the angle between 3 points. p2 is the vertex (Knee).
    Returns the angle in degrees.
    """
    try:
        a = np.array(p1[:2]) # Hip
        b = np.array(p2[:2]) # Knee
        c = np.array(p3[:2]) # Ankle

        ba = a - b
        bc = c - b

        cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
        cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
        angle = np.arccos(cosine_angle)
        return np.degrees(angle)
    except:
        return 0

def main():
    # 1. Load YOLO-Pose model
    model = YOLO('yolo11n-pose.pt')

    # 2. Select Video Source
    if USE_WEBCAM:
        cap = cv2.VideoCapture(0)
        print("--- Starting in WEBCAM Mode ---")
    else:
        cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
        print(f"--- Starting in VIDEO Mode: {INPUT_VIDEO_PATH} ---")

    if not cap.isOpened():
        print("❌ ERROR: Cannot open video source. Check your camera or file path.")
        return

    # 3. Setup VideoWriter to save the output file
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30 # Default fps fallback

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

    # 4. State Variables for Squat Logic
    squat_count = 0
    squat_stage = None # 'DOWN' or 'UP'

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("\n✅ Video processing completed!")
            break

        # Flip frame like a mirror if using webcam
        if USE_WEBCAM:
            frame = cv2.flip(frame, 1)

        # Predict using YOLO-Pose
        results = model.predict(frame, conf=0.5, verbose=False)

        # Default status text
        status_text = "READY"
        status_color = (0, 255, 0)

        for r in results:
            if r.keypoints is not None and len(r.keypoints.data) > 0:
                kpts = r.keypoints.data[0].cpu().numpy()

                try:
                    # Index: 12 (Right Hip), 14 (Right Knee), 16 (Right Ankle)
                    hip = kpts[12]
                    knee = kpts[14]
                    ankle = kpts[16]

                    # Ensure confidence is above 0.5 for all 3 points
                    if hip[2] > 0.5 and knee[2] > 0.5 and ankle[2] > 0.5:
                        angle = calculate_angle(hip, knee, ankle)

                        # LOGIC: Down if angle < 90, Up if angle > 160
                        if angle < 90:
                            squat_stage = "DOWN"
                        if angle > 160 and squat_stage == "DOWN":
                            squat_stage = "UP"
                            squat_count += 1

                        # Update status text
                        status_text = f"SQUATS: {squat_count} ({squat_stage if squat_stage else 'READY'})"
                        status_color = (255, 255, 0) if squat_stage == "DOWN" else (0, 255, 0)

                        # Draw Skeleton Lines (Hip to Knee, Knee to Ankle)
                        cv2.line(frame, (int(hip[0]), int(hip[1])), (int(knee[0]), int(knee[1])), (255, 255, 255), 3)
                        cv2.line(frame, (int(knee[0]), int(knee[1])), (int(ankle[0]), int(ankle[1])), (255, 255, 255), 3)

                        # Draw Points
                        cv2.circle(frame, (int(hip[0]), int(hip[1])), 8, (255, 0, 0), -1)
                        cv2.circle(frame, (int(knee[0]), int(knee[1])), 8, (0, 0, 255), -1)
                        cv2.circle(frame, (int(ankle[0]), int(ankle[1])), 8, (255, 0, 0), -1)

                        # Display Angle text near the knee
                        cv2.putText(frame, f"{int(angle)} deg", (int(knee[0]) + 15, int(knee[1])),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                except Exception as e:
                    pass # Skip if points are missing

        # --- UI Dashboard Display ---
        cv2.rectangle(frame, (10, 10), (400, 70), (0, 0, 0), -1)
        cv2.putText(frame, "CS483: SQUAT COUNTER", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, status_text, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

        # Write frame to output video
        out.write(frame)

        # Display window (only works when running locally, not in Colab)
        try:
            cv2.imshow("CS483: Action Recognition", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        except:
            pass # Ignore if environment doesn't support cv2.imshow

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"🎉 Process finished! Output saved to: {OUTPUT_VIDEO_PATH}")

if __name__ == "__main__":
    main()

--- Starting in VIDEO Mode: input_squat.mp4 ---

✅ Video processing completed!
🎉 Process finished! Output saved to: output_squat_result.mp4
